In [2]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
print("✅ Kaggle installed")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


✅ Kaggle installed


In [3]:
# ==============================================================
# CELL 1 — PHASE 1: KAGGLE CREDENTIALS
# Set KAGGLE_USERNAME and KAGGLE_KEY as environment variables before running
# (or place kaggle.json in ~/.kaggle/). Credentials are never stored in this notebook.
# ==============================================================

import os, json, subprocess

KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
KAGGLE_KEY      = os.environ.get("KAGGLE_KEY", "")

kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")
if KAGGLE_USERNAME and KAGGLE_KEY:
    os.makedirs(os.path.dirname(kaggle_json_path), exist_ok=True)
    with open(kaggle_json_path, "w") as f:
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    os.chmod(kaggle_json_path, 0o600)

result = subprocess.run(["kaggle", "datasets", "list", "--max-size", "1"],
                        capture_output=True, text=True)
print("✅ Kaggle API authenticated" if result.returncode == 0 else f"❌ Auth failed: {result.stderr}")


In [7]:
# ==============================================================
# CELL 2 — PHASE 1: INSTALL PACKAGES
# ==============================================================
 
# %%capture
import subprocess, sys
 
packages = [
    "peft==0.7.0",
    "open_clip_torch==2.23.0",
    "albumentations==1.3.1",
    "grad-cam==1.5.0",
    "timm==0.9.12",
    "scikit-learn",
]
 
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
 
print("✅ All packages installed")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


✅ All packages installed



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [8]:
# ==============================================================
# CELL 3 — PHASE 1: DOWNLOAD & EXTRACT ALL DATASETS
# Creates /workspace/data/ with all 4 datasets
# ==============================================================
 
import os, zipfile, subprocess
 
WORKSPACE    = "/workspace"
DATA_DIR     = f"{WORKSPACE}/data"
WEIGHTS_DIR  = f"{WORKSPACE}/weights"
RESULTS_DIR  = f"{WORKSPACE}/results"
FIGURES_DIR  = f"{WORKSPACE}/figures"
 
for d in [DATA_DIR, WEIGHTS_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)
 
DATASETS = [
    ("kmader/pulmonary-chest-xray-abnormalities", "pulmonary-chest-xray-abnormalities"),
    ("vbookshelf/tbx11k-simplified",              "tbx11k-simplified"),
    ("paultimothymooney/chest-xray-pneumonia",    "chest-xray-pneumonia"),
    ("tawsifurrahman/tuberculosis-tb-chest-xray-dataset", "tuberculosis-tb-chest-xray-dataset"),
]
 
for dataset_slug, folder_name in DATASETS:
    zip_path    = f"{WORKSPACE}/{folder_name}.zip"
    extract_dir = f"{DATA_DIR}/{folder_name}"
 
    if os.path.exists(extract_dir):
        print(f"  ⏭  Already extracted: {folder_name}")
        continue
 
    print(f"  ⬇️  Downloading: {dataset_slug} ...")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", dataset_slug,
         "-p", WORKSPACE, "--unzip"],
        check=True
    )
 
    # kaggle --unzip downloads directly to WORKSPACE; move to DATA_DIR
    # If it zipped anyway, unzip manually:
    if os.path.exists(zip_path):
        print(f"  📦 Extracting: {folder_name} ...")
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)
        os.remove(zip_path)
        print(f"  ✅ Done")
    else:
        # Kaggle already extracted it, just note it
        print(f"  ✅ Extracted directly")
 
print("\n✅ All datasets ready in", DATA_DIR)
 
 

  ⬇️  Downloading: kmader/pulmonary-chest-xray-abnormalities ...
Dataset URL: https://www.kaggle.com/datasets/kmader/pulmonary-chest-xray-abnormalities
License(s): unknown


100%|██████████| 4.09G/4.09G [00:52<00:00, 83.7MB/s]



  ✅ Extracted directly
  ⬇️  Downloading: vbookshelf/tbx11k-simplified ...
Dataset URL: https://www.kaggle.com/datasets/vbookshelf/tbx11k-simplified
License(s): Attribution 4.0 International (CC BY 4.0)


100%|██████████| 2.95G/2.95G [00:37<00:00, 85.1MB/s]



  ✅ Extracted directly
  ⬇️  Downloading: paultimothymooney/chest-xray-pneumonia ...
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other


100%|██████████| 2.29G/2.29G [00:24<00:00, 101MB/s] 



  ✅ Extracted directly
  ⬇️  Downloading: tawsifurrahman/tuberculosis-tb-chest-xray-dataset ...
Dataset URL: https://www.kaggle.com/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset
License(s): copyright-authors


100%|██████████| 663M/663M [00:10<00:00, 63.9MB/s] 



  ✅ Extracted directly

✅ All datasets ready in /workspace/data


In [11]:
DATASET_PATHS = {
    "shenzhen":   "/workspace/ChinaSet_AllFiles/ChinaSet_AllFiles",
    "montgomery": "/workspace/Montgomery/MontgomerySet",
    "tbx11k":     "/workspace/tbx11k-simplified",
    "pneumonia":  "/workspace/chest_xray",
    "tb_dataset": "/workspace/TB_Chest_Radiography_Database",
}

# Verify
import os
print("Dataset path verification:\n")
for name, path in DATASET_PATHS.items():
    if os.path.exists(path):
        n = sum(len(files) for _, _, files in os.walk(path))
        print(f"  ✅ {name:<20s} → {n} files")
    else:
        print(f"  ❌ {name:<20s} → NOT FOUND: {path}")
 


Dataset path verification:

  ✅ shenzhen             → 1326 files
  ✅ montgomery           → 557 files
  ✅ tbx11k               → 11703 files
  ✅ pneumonia            → 17591 files
  ✅ tb_dataset           → 4203 files


In [13]:
import subprocess, sys

packages = [
    "pandas", "numpy", "matplotlib", "seaborn",
    "opencv-python", "Pillow", "tqdm", "scikit-learn",
    "albumentations", "torch", "torchvision",
    "peft==0.7.0", "open_clip_torch==2.23.0",
    "transformers", "timm==0.9.12", "grad-cam==1.5.0",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print("\n✅ All packages installed")

Installing pandas...



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


Installing numpy...



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


Installing matplotlib...



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


... (22 outputs left out)

Installing timm==0.9.12...



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


Installing grad-cam==1.5.0...

✅ All packages installed



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [14]:
# ==============================================================
# CELL 5 — PHASE 2: ALL IMPORTS + GLOBAL SEED
# ==============================================================
 
import os, sys, glob, random, warnings, time, json
from pathlib import Path
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')           # headless — no display needed on RunPod
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
 
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    precision_recall_curve, auc as sklearn_auc,
    roc_curve, confusion_matrix
)
 
import albumentations as A
from albumentations.pytorch import ToTensorV2
 
import open_clip
from transformers import AutoModel
from peft import LoraConfig, get_peft_model
 
warnings.filterwarnings("ignore")
 
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
 
seed_everything(42)
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
 
print("\n✅ Imports done")

PyTorch : 2.4.1+cu124
Device  : cuda
GPU     : NVIDIA A40
VRAM    : 47.7 GB

✅ Imports done


In [18]:
# ==============================================================
# CELL 6 — PHASE 2: GLOBAL HYPERPARAMETERS
# Single source of truth — change here, propagates everywhere
# ==============================================================
 
# ── Training budget (reviewer fix: all configs equal) ─────────
MAX_EPOCHS  = 50
PATIENCE    = 15
BATCH_SIZE  = 64       # A40 48GB handles this fine; drop to 32 if OOM
NUM_WORKERS = 4        # RunPod has 9 vCPU
NUM_FOLDS   = 5
NUM_CLASSES = 2        # binary: TB=0, Normal=1 (after remap)
IMAGE_SIZE  = 224
 
# ── Optimiser LRs ─────────────────────────────────────────────
LR_MAP = {
    "linear_probe": 1e-3,
    "bitfit":       5e-4,
    "lora":         1e-4,
    "full_finetune":1e-5,
}
 
# ── Paths ─────────────────────────────────────────────────────
WORKSPACE   = "/workspace"
DATA_DIR    = f"{WORKSPACE}/data"
WEIGHTS_DIR = f"{WORKSPACE}/weights"
RESULTS_DIR = f"{WORKSPACE}/results"
FIGURES_DIR = f"{WORKSPACE}/figures"
 
DATASET_PATHS = {
    "shenzhen":   "/workspace/ChinaSet_AllFiles/ChinaSet_AllFiles",
    "montgomery": "/workspace/Montgomery/MontgomerySet",
    "tbx11k":     "/workspace/tbx11k-simplified",
    "pneumonia":  "/workspace/chest_xray",
    "tb_dataset": "/workspace/TB_Chest_Radiography_Database",
}

 
# ImageNet normalisation (required by both models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
 
print("✅ Hyperparameters set")
print(f"   MAX_EPOCHS={MAX_EPOCHS} | PATIENCE={PATIENCE} | BATCH_SIZE={BATCH_SIZE}")
 

✅ Hyperparameters set
   MAX_EPOCHS=50 | PATIENCE=15 | BATCH_SIZE=64


In [19]:
# ==============================================================
# CELL 7 — PHASE 2: PREPROCESSING + DATASET
# Key change: apply_clahe is now a parameter (enables ablation)
# ==============================================================
 
def apply_clahe_fn(image, clip_limit=2.0, grid_size=(8, 8)):
    """CLAHE on L-channel of LAB colourspace — standard for CXR."""
    if len(image.shape) == 3:
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size)
    return clahe.apply(image)
 
 
train_transforms = A.Compose([
    A.LongestMaxSize(max_size=IMAGE_SIZE + 32),
    A.RandomResizedCrop(height=IMAGE_SIZE, width=IMAGE_SIZE, scale=(0.85, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    A.GaussNoise(var_limit=(5.0, 15.0), p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
 
val_transforms = A.Compose([
    A.LongestMaxSize(max_size=IMAGE_SIZE),
    A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE,
                  border_mode=cv2.BORDER_CONSTANT, value=0),
    A.CenterCrop(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
 
 
class CXRDataset(Dataset):
    """
    Chest X-ray dataset.
    apply_clahe=True  → standard pipeline (all main experiments)
    apply_clahe=False → ablation to answer Reviewer Q4
    """
 
    def __init__(self, dataframe, transform=None, apply_clahe=True):
        self.df             = dataframe.reset_index(drop=True)
        self.transform      = transform
        self.apply_clahe    = apply_clahe
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.image_path)
        if img is None:
            img = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
 
        if self.apply_clahe:
            img = apply_clahe_fn(img)
 
        if self.transform:
            img = self.transform(image=img)["image"]
 
        return img, torch.tensor(row.label, dtype=torch.long)
 
    def get_labels(self):
        return self.df.label.values
 
 
def create_dataloaders(train_df, val_df, batch_size=BATCH_SIZE,
                       num_workers=NUM_WORKERS, apply_clahe=True):
    train_ds = CXRDataset(train_df, transform=train_transforms, apply_clahe=apply_clahe)
    val_ds   = CXRDataset(val_df,   transform=val_transforms,   apply_clahe=apply_clahe)
 
    labels        = train_ds.get_labels()
    class_counts  = np.bincount(labels)
    class_weights = 1.0 / class_counts
    sample_w      = class_weights[labels]
    sampler       = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
 
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader
 
 
print("✅ CXRDataset + DataLoaders defined")

✅ CXRDataset + DataLoaders defined


In [20]:
# ==============================================================
# CELL 8 — PHASE 2: BUILD UNIFIED DATAFRAME
# Identical logic to original, paths updated for /workspace
# ==============================================================
 
records = []
 
# ── Shenzhen ──────────────────────────────────────────────────
shenzhen_imgs = glob.glob(
    os.path.join(DATASET_PATHS["shenzhen"], "CXR_png", "CHNCXR_*.png")
)
for fpath in shenzhen_imgs:
    digit = os.path.basename(fpath).replace(".png", "").split("_")[-1]
    records.append({
        "image_path": fpath, "label": 0 if digit == "1" else 2,
        "label_name": "TB" if digit == "1" else "Normal",
        "dataset_origin": "shenzhen", "geography": "China",
    })
 
# ── Montgomery ────────────────────────────────────────────────
mont_imgs = glob.glob(
    os.path.join(DATASET_PATHS["montgomery"], "CXR_png", "MCUCXR_*.png")
)
for fpath in mont_imgs:
    digit = os.path.basename(fpath).replace(".png", "").split("_")[-1]
    records.append({
        "image_path": fpath, "label": 0 if digit == "1" else 2,
        "label_name": "TB" if digit == "1" else "Normal",
        "dataset_origin": "montgomery", "geography": "USA",
    })
 
# ── TBX11K ────────────────────────────────────────────────────
# Reviewer Q3: we ONLY keep 'tb' and 'no_tb'; sick-but-non-TB is excluded.
# The composition is logged below for the rebuttal response.
tbx_csv     = os.path.join(DATASET_PATHS["tbx11k"], "data.csv")
tbx_img_dir = os.path.join(DATASET_PATHS["tbx11k"], "images")
tbx_df_raw  = pd.read_csv(tbx_csv)
 
tbx_label_counts = tbx_df_raw["target"].value_counts()
print("TBX11K raw label distribution (for Reviewer Q3 rebuttal):")
print(tbx_label_counts.to_string())
print()
 
tbx_excluded = 0
for _, row in tbx_df_raw.iterrows():
    target = str(row["target"]).strip().lower()
    if target == "tb":
        label, lname = 0, "TB"
    elif target == "no_tb":
        label, lname = 2, "Normal"
    else:
        tbx_excluded += 1
        continue
    img_path = os.path.join(tbx_img_dir, row["fname"])
    if os.path.exists(img_path):
        records.append({
            "image_path": img_path, "label": label, "label_name": lname,
            "dataset_origin": "tbx11k", "geography": "China",
        })
 
print(f"TBX11K: excluded {tbx_excluded} 'sick-non-TB' images from negative class.")
print(f"        Negative class = confirmed healthy only (no_tb).")
 
# ── Kaggle TB ─────────────────────────────────────────────────
for cls_dir, label, lname in [
    ("Normal",       2, "Normal"),
    ("Tuberculosis", 0, "TB"),
]:
    folder = os.path.join(DATASET_PATHS["tb_dataset"], cls_dir)
    if os.path.exists(folder):
        for f in os.listdir(folder):
            if f.lower().endswith((".png", ".jpg", ".jpeg")) and not f.startswith("."):
                records.append({
                    "image_path": os.path.join(folder, f),
                    "label": label, "label_name": lname,
                    "dataset_origin": "kaggle_tb", "geography": "Multi-source",
                })
 
# ── Build DataFrame ───────────────────────────────────────────
df = pd.DataFrame(records)
print("\nDataset summary:")
print(df.groupby(["dataset_origin", "label_name"]).size().unstack(fill_value=0).to_string())
 
# Shenzhen subset with 5-fold splits for training
df_shenzhen = df[df.dataset_origin == "shenzhen"].copy()
 
skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
df_shenzhen["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(
        skf.split(df_shenzhen, df_shenzhen["label"])):
    df_shenzhen.iloc[val_idx, df_shenzhen.columns.get_loc("fold")] = fold_idx
 
print(f"\nShenzhen folds ({len(df_shenzhen)} images):")
for fold in range(NUM_FOLDS):
    n_train = (df_shenzhen.fold != fold).sum()
    n_val   = (df_shenzhen.fold == fold).sum()
    print(f"  Fold {fold}: train={n_train}, val={n_val}")
 
# Remap all labels to 0/1 binary for training
# TB=0 stays 0, Normal=2 → 1
label_remap = {0: 0, 2: 1}
df_shenzhen["label"] = df_shenzhen["label"].map(label_remap)
 
print("\n✅ Unified DataFrame ready")

TBX11K raw label distribution (for Reviewer Q3 rebuttal):
target
no_tb    7600
tb       1211

TBX11K: excluded 0 'sick-non-TB' images from negative class.
        Negative class = confirmed healthy only (no_tb).

Dataset summary:
label_name      Normal    TB
dataset_origin              
kaggle_tb         3500   700
montgomery          80    58
shenzhen           326   336
tbx11k            7600  1211

Shenzhen folds (662 images):
  Fold 0: train=529, val=133
  Fold 1: train=529, val=133
  Fold 2: train=530, val=132
  Fold 3: train=530, val=132
  Fold 4: train=530, val=132

✅ Unified DataFrame ready


In [21]:
# ==============================================================
# CELL 9 — PHASE 2: MODEL CLASSES
# Unchanged from original — backbone wrappers only
# ==============================================================
 
class BiomedCLIPClassifier(nn.Module):
 
    def __init__(self, num_classes=2, freeze_backbone=False):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(
            "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
        )
        self.backbone = clip_model.visual
 
        # Probe embed dim
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224)
            out = self.backbone(dummy)
            out = out[0] if isinstance(out, tuple) else out
            self.embed_dim = out.shape[-1]
 
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
 
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Dropout(0.2),
            nn.Linear(self.embed_dim, num_classes),
        )
 
    def forward(self, x):
        feat = self.backbone(x)
        feat = feat[0] if isinstance(feat, tuple) else feat
        return self.classifier(feat)
 
    def get_features(self, x):
        with torch.no_grad():
            feat = self.backbone(x)
            feat = feat[0] if isinstance(feat, tuple) else feat
        return feat
 
 
class RadDINOClassifier(nn.Module):
 
    def __init__(self, num_classes=2, freeze_backbone=False):
        super().__init__()
        self.backbone  = AutoModel.from_pretrained("microsoft/rad-dino")
        self.embed_dim = self.backbone.config.hidden_size  # 768
 
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
 
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Dropout(0.2),
            nn.Linear(self.embed_dim, num_classes),
        )
 
    def forward(self, x):
        outputs = self.backbone(x)
        cls = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls)
 
    def get_features(self, x):
        with torch.no_grad():
            return self.backbone(x).last_hidden_state[:, 0, :]
 
 
print("✅ BiomedCLIPClassifier and RadDINOClassifier defined")
 

✅ BiomedCLIPClassifier and RadDINOClassifier defined


In [22]:
# ==============================================================
# CELL 10 — PHASE 2: UPDATED MODEL FACTORY
#
# Changes vs original:
#   1. RAD-DINO LoRA targets: "query","key","value","dense"
#      (was "query","value" — fixes Reviewer Q2)
#   2. BitFit method added (addresses Reviewer Q9)
#   3. print_trainable_params now returns dict (used by run_experiment)
# ==============================================================
 
def get_lora_target_modules(model_name):
    """
    Equalized LoRA targets — Reviewer Q2 fix.
    Both models now cover Q, K, V and the output projection,
    ensuring structural parity in adaptation capacity.
    """
    if model_name == "biomedclip":
        # ViT-B/16 open_clip: fused QKV projection + output proj
        return ["qkv", "proj"]
    elif model_name == "rad_dino":
        # HuggingFace ViT-B/14: separate Q/K/V + dense (output proj)
        # EQUALIZED to match BiomedCLIP's Q+K+V+Proj coverage
        return ["query", "key", "value", "dense"]
    raise ValueError(f"Unknown model: {model_name}")
 
 
def apply_lora(model, model_name, rank=8, alpha=None):
    if alpha is None:
        alpha = rank * 2
    config = LoraConfig(
        r=rank,
        lora_alpha=alpha,
        target_modules=get_lora_target_modules(model_name),
        lora_dropout=0.1,
        bias="none",
    )
    model.backbone = get_peft_model(model.backbone, config)
    return model
 
 
def apply_bitfit(model):
    """
    BitFit: freeze everything, unfreeze bias terms only.
    Alternative PEFT baseline — addresses Reviewer Q9.
    """
    for name, param in model.backbone.named_parameters():
        param.requires_grad = ("bias" in name)
    return model
 
 
def print_trainable_params(model, label=""):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    pct       = 100 * trainable / total
    tag       = f"[{label}] " if label else ""
    print(f"  {tag}Trainable: {trainable:>10,} / {total:,}  ({pct:.4f}%)")
    return trainable, total
 
 
def create_model(model_name, method, num_classes=NUM_CLASSES, lora_rank=8):
    """
    Factory for all model × method combinations.
 
    model_name : 'biomedclip' | 'rad_dino'
    method     : 'linear_probe' | 'bitfit' | 'lora' | 'full_finetune'
    lora_rank  : 4 | 8 | 16  (only used when method='lora')
    """
    # ── Instantiate backbone + head ───────────────────────────
    cls = BiomedCLIPClassifier if model_name == "biomedclip" else RadDINOClassifier
 
    if method == "linear_probe":
        model = cls(num_classes=num_classes, freeze_backbone=True)
 
    elif method == "bitfit":
        model = cls(num_classes=num_classes, freeze_backbone=False)
        # Freeze everything first, then selectively unfreeze biases
        for p in model.backbone.parameters():
            p.requires_grad = False
        model = apply_bitfit(model)
 
    elif method == "lora":
        model = cls(num_classes=num_classes, freeze_backbone=False)
        for p in model.backbone.parameters():
            p.requires_grad = False
        model = apply_lora(model, model_name, rank=lora_rank)
 
    elif method == "full_finetune":
        model = cls(num_classes=num_classes, freeze_backbone=False)
 
    else:
        raise ValueError(f"Unknown method: {method}")
 
    # Classification head is always trainable
    for p in model.classifier.parameters():
        p.requires_grad = True
 
    return model
 
 
# Quick factory verification
print("Model Factory — parameter counts:\n")
print(f"{'Configuration':<45} {'Trainable':>12}  {'%':>8}")
print("─" * 68)
 
test_configs = [
    ("biomedclip", "linear_probe", None),
    ("biomedclip", "bitfit",       None),
    ("biomedclip", "lora",            4),
    ("biomedclip", "lora",            8),
    ("biomedclip", "lora",           16),
    ("biomedclip", "full_finetune", None),
    ("rad_dino",   "linear_probe", None),
    ("rad_dino",   "bitfit",       None),
    ("rad_dino",   "lora",            4),
    ("rad_dino",   "lora",            8),
    ("rad_dino",   "lora",           16),
    ("rad_dino",   "full_finetune", None),
]
 
for mn, mt, rank in test_configs:
    m = create_model(mn, mt, lora_rank=rank or 8)
    tr = sum(p.numel() for p in m.parameters() if p.requires_grad)
    to = sum(p.numel() for p in m.parameters())
    label = f"{mn}/{mt}" + (f" r={rank}" if rank else "")
    print(f"  {label:<43} {tr:>12,}  {100*tr/to:>7.4f}%")
    del m
    torch.cuda.empty_cache()
 
print("\n✅ Model factory verified")


Model Factory — parameter counts:

Configuration                                    Trainable         %
────────────────────────────────────────────────────────────────────


open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

  biomedclip/linear_probe                            2,050   0.0024%
  biomedclip/bitfit                                104,962   0.1218%
  biomedclip/lora r=4                              234,498   0.2713%
  biomedclip/lora r=8                              466,946   0.5388%
  biomedclip/lora r=16                             931,842   1.0696%
  biomedclip/full_finetune                      86,193,922  100.0000%


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

... (9 outputs left out)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  rad_dino/lora r=16                             1,182,722   1.3476%


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  rad_dino/full_finetune                        86,583,554  100.0000%

✅ Model factory verified


In [23]:
# ==============================================================
# CELL 11 — PHASE 2: LOSS + EARLY STOPPING
# ==============================================================
 
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight
 
    def forward(self, inputs, targets):
        ce   = F.cross_entropy(inputs, targets, weight=self.weight, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()
 
 
class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=0.001):
        self.patience    = patience
        self.min_delta   = min_delta
        self.counter     = 0
        self.best_score  = None
        self.should_stop = False
 
    def __call__(self, score):
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop
 
    def reset(self):
        self.counter     = 0
        self.best_score  = None
        self.should_stop = False
 
 
print("✅ FocalLoss and EarlyStopping defined")
 

✅ FocalLoss and EarlyStopping defined


In [24]:
# ==============================================================
# CELL 12 — PHASE 2: TRAINING + EVALUATION FUNCTIONS
#
# Changes vs original:
#   1. train_one_epoch: VRAM tracking added (Reviewer Q10)
#   2. train_one_epoch: per-epoch history returned (for Fig 2)
#   3. evaluate: PR-AUC added (Reviewer Q7)
#   4. evaluate: Sensitivity + Specificity at fixed Spec=90%/95%
#      added (Reviewer Q7, clinical metrics)
# ==============================================================
 
def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    """Train one epoch. Returns (loss, auroc, acc, peak_vram_gb)."""
    model.train()
 
    # ── VRAM tracking (Reviewer Q10) ──────────────────────────
    torch.cuda.reset_peak_memory_stats(device)
 
    running_loss  = 0.0
    all_preds, all_labels = [], []
 
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
 
        with autocast():
            outputs = model(images)
            loss    = criterion(outputs, labels)
 
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
 
        running_loss += loss.item() * images.size(0)
        all_preds.append(torch.softmax(outputs.detach(), dim=1).cpu())
        all_labels.append(labels.cpu())
 
    peak_vram = torch.cuda.max_memory_allocated(device) / 1e9   # GB
 
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    epoch_loss = running_loss / len(loader.dataset)
 
    try:
        auroc = roc_auc_score(all_labels, all_preds[:, 1])
    except ValueError:
        auroc = 0.0
 
    acc = accuracy_score(all_labels, all_preds.argmax(axis=1))
    return epoch_loss, auroc, acc, peak_vram
 
 
def sensitivity_specificity_at_fixed_spec(y_true, y_prob, target_spec=0.95):
    """
    Find the operating point where Specificity >= target_spec,
    then report Sensitivity at that point.
    Addresses Reviewer Q7: 'sensitivity/specificity at clinically
    meaningful operating points'.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob, pos_label=1)
    spec = 1 - fpr                  # specificity = 1 - FPR
 
    # Find threshold where specificity is closest to (but >= ) target
    valid = spec >= target_spec
    if not valid.any():
        return np.nan, np.nan, np.nan
 
    idx        = np.where(valid)[0][-1]   # highest sensitivity at target spec
    sens_at    = tpr[idx]
    spec_at    = spec[idx]
    thresh_at  = thresholds[idx]
    return float(sens_at), float(spec_at), float(thresh_at)
 
 
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    Full evaluation.
    Returns dict with: loss, auroc, accuracy, f1,
                       pr_auc, sens_at_90spec, sens_at_95spec,
                       predictions, labels
    """
    model.eval()
    running_loss  = 0.0
    all_preds, all_labels = [], []
 
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with autocast():
            outputs = model(images)
            loss    = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        all_preds.append(torch.softmax(outputs, dim=1).cpu())
        all_labels.append(labels.cpu())
 
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    epoch_loss = running_loss / len(loader.dataset)
 
    pos_probs = all_preds[:, 1]      # probability of TB=1 after remap
 
    # ── Core metrics ──────────────────────────────────────────
    try:
        auroc = roc_auc_score(all_labels, pos_probs)
    except ValueError:
        auroc = 0.0
 
    acc = accuracy_score(all_labels, all_preds.argmax(axis=1))
    f1  = f1_score(all_labels, all_preds.argmax(axis=1), average="macro", zero_division=0)
 
    # ── PR-AUC (Reviewer Q7) ──────────────────────────────────
    try:
        prec, rec, _ = precision_recall_curve(all_labels, pos_probs)
        pr_auc       = sklearn_auc(rec, prec)
    except ValueError:
        pr_auc = 0.0
 
    # ── Sensitivity at fixed Specificity (Reviewer Q7) ────────
    sens_90, spec_90, _ = sensitivity_specificity_at_fixed_spec(
        all_labels, pos_probs, target_spec=0.90
    )
    sens_95, spec_95, _ = sensitivity_specificity_at_fixed_spec(
        all_labels, pos_probs, target_spec=0.95
    )
 
    return {
        "loss":        epoch_loss,
        "auroc":       auroc,
        "accuracy":    acc,
        "f1":          f1,
        "pr_auc":      pr_auc,
        "sens_at_90spec": sens_90,
        "sens_at_95spec": sens_95,
        "predictions": all_preds,
        "labels":      all_labels,
    }
 
 
print("✅ train_one_epoch (+ VRAM) and evaluate (+ PR-AUC, Sens/Spec) defined")
 
 

✅ train_one_epoch (+ VRAM) and evaluate (+ PR-AUC, Sens/Spec) defined


In [25]:
# ==============================================================
# CELL 13 — PHASE 2: RUN_EXPERIMENT
#
# Changes vs original:
#   1. Saves model weights to disk after each fold (CRITICAL for
#      Phase 4 cross-dataset evaluation — Reviewer Q1)
#   2. Logs per-epoch history (loss + AUROC) for training curves
#   3. Records peak_vram per epoch (Reviewer Q10)
#   4. apply_clahe parameter passed through (CLAHE ablation)
# ==============================================================
 
def run_experiment(
    model_name,
    method,
    train_df,
    val_df,
    fold,
    num_classes  = NUM_CLASSES,
    lora_rank    = 8,
    batch_size   = BATCH_SIZE,
    max_epochs   = MAX_EPOCHS,
    patience     = PATIENCE,
    apply_clahe  = True,
    device       = None,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
    # ── Label remap to contiguous 0/1 ─────────────────────────
    unique_labels = sorted(train_df.label.unique())
    label_remap   = {old: new for new, old in enumerate(unique_labels)}
    train_df      = train_df.copy()
    val_df        = val_df.copy()
    train_df["label"] = train_df["label"].map(label_remap)
    val_df["label"]   = val_df["label"].map(label_remap)
 
    # ── Learning rate ─────────────────────────────────────────
    lr = LR_MAP.get(method, 1e-4)
 
    # ── Build model ───────────────────────────────────────────
    model      = create_model(model_name, method, num_classes=num_classes, lora_rank=lora_rank)
    model      = model.to(device)
    trainable, total = print_trainable_params(model, label=f"{model_name}/{method}")
 
    # ── DataLoaders ───────────────────────────────────────────
    train_loader, val_loader = create_dataloaders(
        train_df, val_df, batch_size=batch_size, apply_clahe=apply_clahe
    )
 
    # ── Class-weighted Focal Loss ─────────────────────────────
    labels_arr   = train_df.label.values
    class_counts = np.bincount(labels_arr, minlength=num_classes).astype(float)
    cw           = torch.FloatTensor(1.0 / class_counts).to(device)
    cw           = cw / cw.sum() * num_classes
    criterion    = FocalLoss(gamma=2.0, weight=cw)
 
    # ── Optimiser + Scheduler ─────────────────────────────────
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=0.01
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max_epochs, eta_min=lr * 0.01
    )
    scaler        = GradScaler()
    early_stop    = EarlyStopping(patience=patience)
 
    # ── Training loop ─────────────────────────────────────────
    best_auroc    = 0.0
    best_state    = None
    best_val_res  = None
    history       = []
    start_time    = time.time()
 
    for epoch in range(max_epochs):
        tr_loss, tr_auroc, tr_acc, vram = train_one_epoch(
            model, train_loader, optimizer, criterion, device, scaler
        )
        val_res = evaluate(model, val_loader, criterion, device)
        scheduler.step()
 
        history.append({
            "epoch":      epoch + 1,
            "train_loss": tr_loss,
            "train_auroc":tr_auroc,
            "val_loss":   val_res["loss"],
            "val_auroc":  val_res["auroc"],
            "peak_vram":  vram,
        })
 
        if val_res["auroc"] > best_auroc:
            best_auroc   = val_res["auroc"]
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_val_res = val_res
 
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Ep {epoch+1:3d}/{max_epochs} | "
                  f"TrLoss={tr_loss:.4f} TrAUC={tr_auroc:.4f} | "
                  f"VaLoss={val_res['loss']:.4f} VaAUC={val_res['auroc']:.4f} | "
                  f"VRAM={vram:.2f}GB")
 
        if early_stop(val_res["auroc"]):
            print(f"  ⏹  Early stopping at epoch {epoch+1}")
            break
 
    elapsed = time.time() - start_time
    model.load_state_dict(best_state)
 
    # ── Save weights to disk (CRITICAL for Phase 4) ───────────
    clahe_tag    = "" if apply_clahe else "_no_clahe"
    rank_tag     = f"_r{lora_rank}" if method == "lora" else ""
    weight_fname = f"{model_name}_{method}{rank_tag}{clahe_tag}_fold{fold}.pth"
    weight_path  = os.path.join(WEIGHTS_DIR, weight_fname)
    torch.save(best_state, weight_path)
    print(f"  💾 Weights saved → {weight_path}")
 
    print(f"  ✅ Done in {elapsed/60:.1f} min | Best val AUROC: {best_auroc:.4f}")
 
    # ── Return results ────────────────────────────────────────
    result = {
        "model_name":    model_name,
        "method":        method,
        "lora_rank":     lora_rank if method == "lora" else None,
        "apply_clahe":   apply_clahe,
        "fold":          fold,
        "num_classes":   num_classes,
        "train_size":    len(train_df),
        "val_size":      len(val_df),
        "trainable_params": trainable,
        "total_params":     total,
        "best_val_auroc":   best_auroc,
        "best_val_pr_auc":  best_val_res["pr_auc"],
        "best_val_sens_90": best_val_res["sens_at_90spec"],
        "best_val_sens_95": best_val_res["sens_at_95spec"],
        "history":          history,
        "training_time_sec":elapsed,
        "epochs_trained":   epoch + 1,
        "peak_vram_gb":     max(h["peak_vram"] for h in history),
        "weight_path":      weight_path,
    }
 
    del model
    torch.cuda.empty_cache()
    return result
 
 
print("✅ run_experiment defined (saves weights + VRAM + PR-AUC + Sens/Spec)")
 


✅ run_experiment defined (saves weights + VRAM + PR-AUC + Sens/Spec)


In [26]:
# ==============================================================
# CELL 14 — PHASE 3: EXPERIMENT MATRIX
# 13 configs × 5 folds = 65 runs
# ==============================================================
 
EXPERIMENTS = [
    # (model_name,    method,          lora_rank, apply_clahe)
    ("biomedclip",  "linear_probe",   None,      True),
    ("biomedclip",  "bitfit",         None,      True),
    ("biomedclip",  "lora",           4,         True),
    ("biomedclip",  "lora",           8,         True),
    ("biomedclip",  "lora",           16,        True),
    ("biomedclip",  "full_finetune",  None,      True),
    ("rad_dino",    "linear_probe",   None,      True),
    ("rad_dino",    "bitfit",         None,      True),
    ("rad_dino",    "lora",           4,         True),
    ("rad_dino",    "lora",           8,         True),
    ("rad_dino",    "lora",           16,        True),
    ("rad_dino",    "full_finetune",  None,      True),
    # CLAHE ablation (Reviewer Q4) — RAD-DINO LoRA r=16 only
    ("rad_dino",    "lora",           16,        False),
]
 
total_runs = len(EXPERIMENTS) * NUM_FOLDS
print(f"Experiment matrix: {len(EXPERIMENTS)} configs × {NUM_FOLDS} folds = {total_runs} runs")
print(f"Hyperparameters:   MAX_EPOCHS={MAX_EPOCHS}, PATIENCE={PATIENCE}, BATCH_SIZE={BATCH_SIZE}\n")
print(f"{'#':<3} {'Config':<45} {'CLAHE'}")
print("─" * 58)
for i, (mn, mt, rank, clahe) in enumerate(EXPERIMENTS):
    label = f"{mn}/{mt}" + (f" r={rank}" if rank else "")
    print(f"{i+1:<3} {label:<45} {'✅' if clahe else '❌ ABLATION'}")
 

Experiment matrix: 13 configs × 5 folds = 65 runs
Hyperparameters:   MAX_EPOCHS=50, PATIENCE=15, BATCH_SIZE=64

#   Config                                        CLAHE
──────────────────────────────────────────────────────────
1   biomedclip/linear_probe                       ✅
2   biomedclip/bitfit                             ✅
3   biomedclip/lora r=4                           ✅
4   biomedclip/lora r=8                           ✅
5   biomedclip/lora r=16                          ✅
6   biomedclip/full_finetune                      ✅
7   rad_dino/linear_probe                         ✅
8   rad_dino/bitfit                               ✅
9   rad_dino/lora r=4                             ✅
10  rad_dino/lora r=8                             ✅
11  rad_dino/lora r=16                            ✅
12  rad_dino/full_finetune                        ✅
13  rad_dino/lora r=16                            ❌ ABLATION


In [27]:
# ==============================================================
# CELL 15 — PHASE 3: RUN ALL EXPERIMENTS
# Results checkpointed to JSON after every fold so a crash
# doesn't lose completed work.
# ==============================================================
 
import json
 
RESULTS_FILE = f"{RESULTS_DIR}/experiment_results.json"
 
# Load any previously completed results (crash-safe resume)
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        all_results = json.load(f)
    print(f"▶  Resuming — {len(all_results)} runs already complete")
else:
    all_results = []
 
def already_done(model_name, method, lora_rank, fold, apply_clahe):
    for r in all_results:
        if (r["model_name"]  == model_name  and
            r["method"]      == method       and
            r["lora_rank"]   == lora_rank    and
            r["fold"]        == fold         and
            r["apply_clahe"] == apply_clahe):
            return True
    return False
 
 
for config_idx, (model_name, method, lora_rank, apply_clahe) in enumerate(EXPERIMENTS):
    clahe_tag = "" if apply_clahe else " [NO CLAHE]"
    rank_tag  = f" r={lora_rank}" if lora_rank else ""
    config_label = f"{model_name}/{method}{rank_tag}{clahe_tag}"
 
    print(f"\n{'='*60}")
    print(f"Config {config_idx+1}/{len(EXPERIMENTS)}: {config_label}")
    print(f"{'='*60}")
 
    for fold in range(NUM_FOLDS):
        if already_done(model_name, method, lora_rank, fold, apply_clahe):
            print(f"  ⏭  Fold {fold} already complete — skipping")
            continue
 
        print(f"\n  ── Fold {fold} ──────────────────────────────")
        train_fold = df_shenzhen[df_shenzhen.fold != fold].copy()
        val_fold   = df_shenzhen[df_shenzhen.fold == fold].copy()
 
        result = run_experiment(
            model_name  = model_name,
            method      = method,
            train_df    = train_fold,
            val_df      = val_fold,
            fold        = fold,
            lora_rank   = lora_rank if lora_rank else 8,
            apply_clahe = apply_clahe,
            device      = device,
        )
 
        # Strip non-serialisable arrays before saving to JSON
        result_save = {k: v for k, v in result.items()
                       if k not in ("history",)}
        result_save["history"] = [
            {kk: (float(vv) if isinstance(vv, (np.floating, float)) else vv)
             for kk, vv in h.items()}
            for h in result["history"]
        ]
        all_results.append(result_save)
 
        with open(RESULTS_FILE, "w") as f:
            json.dump(all_results, f, indent=2, default=str)
        print(f"  📝 Checkpoint saved ({len(all_results)} total runs)")
 
print("\n" + "="*60)
print("✅ ALL EXPERIMENTS COMPLETE")
print("="*60)
 
# Quick summary table
results_df = pd.DataFrame(all_results)
summary = (
    results_df
    .groupby(["model_name", "method", "lora_rank", "apply_clahe"])
    .agg(
        mean_auroc  =("best_val_auroc",   "mean"),
        std_auroc   =("best_val_auroc",   "std"),
        mean_pr_auc =("best_val_pr_auc",  "mean"),
        mean_vram   =("peak_vram_gb",     "mean"),
        trainable   =("trainable_params", "first"),
    )
    .reset_index()
    .sort_values("mean_auroc", ascending=False)
)
 
print("\nRESULTS SUMMARY (sorted by AUROC):\n")
print(f"{'Configuration':<45} {'AUROC':>8} {'±':>3} {'PR-AUC':>8} {'VRAM':>7}")
print("─" * 75)
for _, row in summary.iterrows():
    label = f"{row.model_name}/{row.method}"
    if row.lora_rank:
        label += f" r={int(row.lora_rank)}"
    if not row.apply_clahe:
        label += " [no CLAHE]"
    print(f"  {label:<43} {row.mean_auroc:.4f}  ±{row.std_auroc:.4f}  "
          f"{row.mean_pr_auc:.4f}  {row.mean_vram:.2f}GB")
 
summary.to_csv(f"{RESULTS_DIR}/results_summary.csv", index=False)
results_df.to_csv(f"{RESULTS_DIR}/results_all_folds.csv", index=False)
print(f"\n✅ Summary saved to {RESULTS_DIR}/")
print("\n⏭  Proceed to Phase 4: Cross-Dataset Evaluation (next notebook section)")
 


Config 1/13: biomedclip/linear_probe

  ── Fold 0 ──────────────────────────────
  [biomedclip/linear_probe] Trainable:      2,050 / 86,193,922  (0.0024%)
  Ep   1/50 | TrLoss=0.2165 TrAUC=0.5701 | VaLoss=0.1589 VaAUC=0.7386 | VRAM=0.71GB
  Ep   5/50 | TrLoss=0.1312 TrAUC=0.8379 | VaLoss=0.1482 VaAUC=0.8458 | VRAM=0.71GB
  Ep  10/50 | TrLoss=0.1098 TrAUC=0.8782 | VaLoss=0.1405 VaAUC=0.8424 | VRAM=0.71GB
  Ep  15/50 | TrLoss=0.1088 TrAUC=0.8822 | VaLoss=0.1909 VaAUC=0.8373 | VRAM=0.71GB
  Ep  20/50 | TrLoss=0.1023 TrAUC=0.8942 | VaLoss=0.1335 VaAUC=0.8452 | VRAM=0.71GB
  Ep  25/50 | TrLoss=0.0991 TrAUC=0.9035 | VaLoss=0.1421 VaAUC=0.8503 | VRAM=0.71GB
  Ep  30/50 | TrLoss=0.0892 TrAUC=0.9229 | VaLoss=0.1225 VaAUC=0.8520 | VRAM=0.71GB
  Ep  35/50 | TrLoss=0.1149 TrAUC=0.8661 | VaLoss=0.1417 VaAUC=0.8561 | VRAM=0.71GB
  Ep  40/50 | TrLoss=0.0977 TrAUC=0.9063 | VaLoss=0.1467 VaAUC=0.8600 | VRAM=0.71GB
  Ep  45/50 | TrLoss=0.0898 TrAUC=0.9239 | VaLoss=0.1390 VaAUC=0.8596 | VRAM=0.71GB
  Ep

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  [rad_dino/linear_probe] Trainable:      3,074 / 86,583,554  (0.0036%)
  Ep   1/50 | TrLoss=0.1676 TrAUC=0.7511 | VaLoss=0.1097 VaAUC=0.8941 | VRAM=0.91GB
  Ep   5/50 | TrLoss=0.0628 TrAUC=0.9625 | VaLoss=0.0887 VaAUC=0.9386 | VRAM=0.91GB
  Ep  10/50 | TrLoss=0.0488 TrAUC=0.9789 | VaLoss=0.0803 VaAUC=0.9494 | VRAM=0.91GB
  Ep  15/50 | TrLoss=0.0436 TrAUC=0.9812 | VaLoss=0.0872 VaAUC=0.9464 | VRAM=0.91GB
  Ep  20/50 | TrLoss=0.0319 TrAUC=0.9921 | VaLoss=0.0900 VaAUC=0.9452 | VRAM=0.91GB
  Ep  25/50 | TrLoss=0.0446 TrAUC=0.9824 | VaLoss=0.1018 VaAUC=0.9434 | VRAM=0.91GB
  ⏹  Early stopping at epoch 27
  💾 Weights saved → /workspace/weights/rad_dino_linear_probe_fold0.pth
  ✅ Done in 18.7 min | Best val AUROC: 0.9528
  📝 Checkpoint saved (31 total runs)

  ── Fold 1 ──────────────────────────────


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  [rad_dino/linear_probe] Trainable:      3,074 / 86,583,554  (0.0036%)
  Ep   1/50 | TrLoss=0.2457 TrAUC=0.6147 | VaLoss=0.1457 VaAUC=0.8258 | VRAM=0.91GB
  Ep   5/50 | TrLoss=0.0644 TrAUC=0.9612 | VaLoss=0.0781 VaAUC=0.9496 | VRAM=0.91GB
  Ep  10/50 | TrLoss=0.0587 TrAUC=0.9681 | VaLoss=0.0756 VaAUC=0.9525 | VRAM=0.91GB
  Ep  15/50 | TrLoss=0.0505 TrAUC=0.9779 | VaLoss=0.0777 VaAUC=0.9551 | VRAM=0.91GB
  Ep  20/50 | TrLoss=0.0423 TrAUC=0.9833 | VaLoss=0.0784 VaAUC=0.9562 | VRAM=0.91GB
  Ep  25/50 | TrLoss=0.0400 TrAUC=0.9854 | VaLoss=0.0846 VaAUC=0.9557 | VRAM=0.91GB
  Ep  30/50 | TrLoss=0.0399 TrAUC=0.9859 | VaLoss=0.0786 VaAUC=0.9582 | VRAM=0.91GB
  Ep  35/50 | TrLoss=0.0332 TrAUC=0.9912 | VaLoss=0.0805 VaAUC=0.9564 | VRAM=0.91GB
  Ep  40/50 | TrLoss=0.0327 TrAUC=0.9906 | VaLoss=0.0820 VaAUC=0.9570 | VRAM=0.91GB
  Ep  45/50 | TrLoss=0.0340 TrAUC=0.9899 | VaLoss=0.0822 VaAUC=0.9579 | VRAM=0.91GB
  ⏹  Early stopping at epoch 46
  💾 Weights saved → /workspace/weights/rad_dino_linear_p

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

... (61 outputs left out)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  [rad_dino/lora] Trainable:  1,182,722 / 87,763,202  (1.3476%)
  Ep   1/50 | TrLoss=0.2849 TrAUC=0.5308 | VaLoss=0.2124 VaAUC=0.6320 | VRAM=6.35GB
  Ep   5/50 | TrLoss=0.1415 TrAUC=0.8245 | VaLoss=0.0919 VaAUC=0.9272 | VRAM=6.35GB
  Ep  10/50 | TrLoss=0.0582 TrAUC=0.9696 | VaLoss=0.0537 VaAUC=0.9776 | VRAM=6.35GB
  Ep  15/50 | TrLoss=0.0344 TrAUC=0.9887 | VaLoss=0.0437 VaAUC=0.9844 | VRAM=6.35GB
  Ep  20/50 | TrLoss=0.0220 TrAUC=0.9957 | VaLoss=0.0443 VaAUC=0.9846 | VRAM=6.35GB
  Ep  25/50 | TrLoss=0.0129 TrAUC=0.9990 | VaLoss=0.0474 VaAUC=0.9828 | VRAM=6.35GB
  Ep  30/50 | TrLoss=0.0125 TrAUC=0.9988 | VaLoss=0.0474 VaAUC=0.9837 | VRAM=6.35GB
  ⏹  Early stopping at epoch 30
  💾 Weights saved → /workspace/weights/rad_dino_lora_r16_no_clahe_fold3.pth
  ✅ Done in 9.4 min | Best val AUROC: 0.9853
  📝 Checkpoint saved (64 total runs)

  ── Fold 4 ──────────────────────────────


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  [rad_dino/lora] Trainable:  1,182,722 / 87,763,202  (1.3476%)
  Ep   1/50 | TrLoss=0.2061 TrAUC=0.7082 | VaLoss=0.1798 VaAUC=0.7661 | VRAM=6.35GB
  Ep   5/50 | TrLoss=0.0995 TrAUC=0.9065 | VaLoss=0.1061 VaAUC=0.9196 | VRAM=6.35GB
  Ep  10/50 | TrLoss=0.0502 TrAUC=0.9777 | VaLoss=0.0678 VaAUC=0.9699 | VRAM=6.35GB
  Ep  15/50 | TrLoss=0.0230 TrAUC=0.9953 | VaLoss=0.0718 VaAUC=0.9794 | VRAM=6.35GB
  Ep  20/50 | TrLoss=0.0186 TrAUC=0.9968 | VaLoss=0.0620 VaAUC=0.9846 | VRAM=6.35GB
  Ep  25/50 | TrLoss=0.0150 TrAUC=0.9980 | VaLoss=0.0714 VaAUC=0.9844 | VRAM=6.35GB
  Ep  30/50 | TrLoss=0.0135 TrAUC=0.9984 | VaLoss=0.0697 VaAUC=0.9844 | VRAM=6.35GB
  ⏹  Early stopping at epoch 34
  💾 Weights saved → /workspace/weights/rad_dino_lora_r16_no_clahe_fold4.pth
  ✅ Done in 11.4 min | Best val AUROC: 0.9851
  📝 Checkpoint saved (65 total runs)

✅ ALL EXPERIMENTS COMPLETE

RESULTS SUMMARY (sorted by AUROC):

Configuration                                    AUROC   ±   PR-AUC    VRAM
────────────────

In [28]:
import json
with open("/workspace/results/experiment_results.json") as f:
    done = json.load(f)
print(f"Completed: {len(done)}/65")
for r in done[-5:]:
    print(f"  {r['model_name']}/{r['method']} fold{r['fold']} → AUROC {r['best_val_auroc']:.4f}")

Completed: 65/65
  rad_dino/lora fold0 → AUROC 0.9873
  rad_dino/lora fold1 → AUROC 0.9842
  rad_dino/lora fold2 → AUROC 0.9910
  rad_dino/lora fold3 → AUROC 0.9853
  rad_dino/lora fold4 → AUROC 0.9851


In [29]:
# ==============================================================
# PHASE 4 — CELL 16: PREPARE EXTERNAL TEST SETS
# ==============================================================
 
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.cuda.amp import autocast
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    precision_recall_curve, auc as sklearn_auc,
    roc_curve, confusion_matrix
)
 
# Reload results
with open(f"{RESULTS_DIR}/experiment_results.json") as f:
    all_results = json.load(f)
results_df = pd.DataFrame(all_results)
print(f"✅ Loaded {len(all_results)} completed runs")
 
# ── Build external test sets (same label remap as training: TB=0→0, Normal=2→1)
label_remap = {0: 0, 2: 1}
 
df_montgomery = df[df.dataset_origin == "montgomery"].copy()
df_montgomery["label"] = df_montgomery["label"].map(label_remap)
df_montgomery = df_montgomery.dropna(subset=["label"])
df_montgomery["label"] = df_montgomery["label"].astype(int)
 
# TBX11K: already filtered to tb/no_tb only in Cell 8
df_tbx11k = df[df.dataset_origin == "tbx11k"].copy()
df_tbx11k["label"] = df_tbx11k["label"].map(label_remap)
df_tbx11k = df_tbx11k.dropna(subset=["label"])
df_tbx11k["label"] = df_tbx11k["label"].astype(int)
 
df_kaggle_tb = df[df.dataset_origin == "kaggle_tb"].copy()
df_kaggle_tb["label"] = df_kaggle_tb["label"].map(label_remap)
df_kaggle_tb = df_kaggle_tb.dropna(subset=["label"])
df_kaggle_tb["label"] = df_kaggle_tb["label"].astype(int)
 
TEST_SETS = {
    "montgomery": df_montgomery,
    "tbx11k":    df_tbx11k,
    "kaggle_tb": df_kaggle_tb,
}
 
print("\nExternal test set composition:")
for name, d in TEST_SETS.items():
    tb_n     = (d.label == 0).sum()
    normal_n = (d.label == 1).sum()
    print(f"  {name:<15} TB={tb_n}, Normal={normal_n}, Total={len(d)}")
 
print("\n✅ External test sets ready")
 

✅ Loaded 65 completed runs

External test set composition:
  montgomery      TB=58, Normal=80, Total=138
  tbx11k          TB=1211, Normal=7600, Total=8811
  kaggle_tb       TB=700, Normal=3500, Total=4200

✅ External test sets ready


In [30]:
# ==============================================================
# PHASE 4 — CELL 17: EVALUATION UTILITIES
# ==============================================================
 
def sensitivity_specificity_at_fixed_spec(y_true, y_prob, target_spec=0.95):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob, pos_label=0)  # TB=0 is positive
    spec = 1 - fpr
    valid = spec >= target_spec
    if not valid.any():
        return np.nan, np.nan
    idx   = np.where(valid)[0][-1]
    return float(tpr[idx]), float(spec[idx])
 
 
def compute_all_metrics(y_true, y_prob_tb):
    """
    y_true    : array of labels (0=TB, 1=Normal)
    y_prob_tb : model's probability for class 0 (TB)
    """
    # AUROC — TB is positive class (label=0)
    try:
        auroc = roc_auc_score((y_true == 0).astype(int), y_prob_tb)
    except ValueError:
        auroc = np.nan
 
    # PR-AUC
    try:
        prec, rec, _ = precision_recall_curve(
            (y_true == 0).astype(int), y_prob_tb
        )
        pr_auc = sklearn_auc(rec, prec)
    except ValueError:
        pr_auc = np.nan
 
    # Sensitivity / Specificity at fixed operating points
    sens_90, spec_90 = sensitivity_specificity_at_fixed_spec(
        y_true, y_prob_tb, target_spec=0.90
    )
    sens_95, spec_95 = sensitivity_specificity_at_fixed_spec(
        y_true, y_prob_tb, target_spec=0.95
    )
 
    # Accuracy + F1
    y_pred = (y_prob_tb >= 0.5).astype(int)   # threshold at 0.5 for TB
    # convert back: pred=1 means TB, pred=0 means Normal
    acc = accuracy_score(y_true, 1 - y_pred)
    f1  = f1_score(
        (y_true == 0).astype(int),
        y_pred,
        average="binary", zero_division=0
    )
 
    return {
        "auroc":       auroc,
        "pr_auc":      pr_auc,
        "sens_at_90spec": sens_90,
        "spec_at_90":     spec_90,
        "sens_at_95spec": sens_95,
        "spec_at_95":     spec_95,
        "accuracy":    acc,
        "f1":          f1,
    }
 
 
@torch.no_grad()
def get_predictions(model, test_df, device, apply_clahe=True, batch_size=64):
    """Run inference on a test set, return (probs, labels)."""
    model.eval()
    model = model.to(device)
 
    test_ds     = CXRDataset(test_df, transform=val_transforms, apply_clahe=apply_clahe)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             num_workers=4, pin_memory=True)
 
    all_probs, all_labels = [], []
    for images, labels in test_loader:
        images = images.to(device)
        with autocast():
            outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
 
    return np.concatenate(all_probs), np.concatenate(all_labels)
 
 
def load_model_from_weights(model_name, method, lora_rank, fold,
                             apply_clahe=True, device=None):
    """Reconstruct model and load saved weights."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
    clahe_tag = "" if apply_clahe else "_no_clahe"
    rank_tag  = f"_r{lora_rank}" if method == "lora" else ""
    fname     = f"{model_name}_{method}{rank_tag}{clahe_tag}_fold{fold}.pth"
    path      = os.path.join(WEIGHTS_DIR, fname)
 
    if not os.path.exists(path):
        print(f"  ⚠️  Weight file not found: {path}")
        return None
 
    model = create_model(model_name, method,
                         num_classes=NUM_CLASSES,
                         lora_rank=lora_rank if lora_rank else 8)
    state = torch.load(path, map_location=device)
    model.load_state_dict(state)
    model = model.to(device)
    model.eval()
    return model
 
 
print("✅ Phase 4 utilities defined")
 

✅ Phase 4 utilities defined


In [ ]:
# ==============================================================
# PHASE 4 — CELL 18: CROSS-DATASET EVALUATION
#
# For each config:
#   1. Load all 5 fold checkpoints
#   2. Run inference on each external test set
#   3. Report per-fold metrics → mean ± std  (answers Reviewer Q1)
#   4. Also report ensemble (averaged probabilities across folds)
# ==============================================================
 
CROSS_CONFIGS = [
    # (model_name,   method,          lora_rank, apply_clahe)
    ("biomedclip", "linear_probe",   None,      True),
    ("biomedclip", "bitfit",         None,      True),
    ("biomedclip", "lora",           4,         True),
    ("biomedclip", "lora",           8,         True),
    ("biomedclip", "lora",           16,        True),
    ("biomedclip", "full_finetune",  None,      True),
    ("rad_dino",   "linear_probe",   None,      True),
    ("rad_dino",   "bitfit",         None,      True),
    ("rad_dino",   "lora",           4,         True),
    ("rad_dino",   "lora",           8,         True),
    ("rad_dino",   "lora",           16,        True),
    ("rad_dino",   "full_finetune",  None,      True),
    ("rad_dino",   "lora",           16,        False),   # CLAHE ablation
]
 
cross_results = []   # per-fold results
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
CROSS_RESULTS_FILE = f"{RESULTS_DIR}/cross_dataset_results.json"
 
# Resume if interrupted
if os.path.exists(CROSS_RESULTS_FILE):
    with open(CROSS_RESULTS_FILE) as f:
        cross_results = json.load(f)
    print(f"▶  Resuming cross-eval — {len(cross_results)} records already done")
 
def cross_already_done(model_name, method, lora_rank, fold, apply_clahe, test_name):
    for r in cross_results:
        if (r["model_name"]  == model_name  and
            r["method"]      == method       and
            r["lora_rank"]   == lora_rank    and
            r["fold"]        == fold         and
            r["apply_clahe"] == apply_clahe  and
            r["test_dataset"]== test_name):
            return True
    return False
 
 
for model_name, method, lora_rank, apply_clahe in CROSS_CONFIGS:
    clahe_tag = "" if apply_clahe else " [NO CLAHE]"
    rank_tag  = f" r={lora_rank}" if lora_rank else ""
    config_label = f"{model_name}/{method}{rank_tag}{clahe_tag}"
 
    print(f"\n{'='*60}")
    print(f"Evaluating: {config_label}")
    print(f"{'='*60}")
 
    for fold in range(NUM_FOLDS):
        # Check if all test sets already done for this fold
        all_done = all(
            cross_already_done(model_name, method, lora_rank,
                               fold, apply_clahe, ts)
            for ts in TEST_SETS
        )
        if all_done:
            print(f"  ⏭  Fold {fold} already evaluated — skipping")
            continue
 
        # Load model
        model = load_model_from_weights(
            model_name, method, lora_rank, fold, apply_clahe, device
        )
        if model is None:
            continue
 
        print(f"\n  Fold {fold}:")
        for test_name, test_df in TEST_SETS.items():
            if cross_already_done(model_name, method, lora_rank,
                                   fold, apply_clahe, test_name):
                print(f"    ⏭  {test_name} already done")
                continue
 
            probs, labels = get_predictions(
                model, test_df, device,
                apply_clahe=apply_clahe
            )
 
            # probs[:,0] = P(TB), probs[:,1] = P(Normal)
            metrics = compute_all_metrics(labels, probs[:, 0])
 
            record = {
                "model_name":  model_name,
                "method":      method,
                "lora_rank":   lora_rank,
                "apply_clahe": apply_clahe,
                "fold":        fold,
                "test_dataset":test_name,
                **{k: float(v) if not np.isnan(v) else None
                   for k, v in metrics.items()},
            }
            cross_results.append(record)
            print(f"    {test_name:<15} AUROC={metrics['auroc']:.4f}  "
                  f"PR-AUC={metrics['pr_auc']:.4f}  "
                  f"Sens@90spec={metrics['sens_at_90spec']:.3f}")
 
        # Save checkpoint after each fold
        with open(CROSS_RESULTS_FILE, "w") as f:
            json.dump(cross_results, f, indent=2, default=str)
 
        del model
        torch.cuda.empty_cache()
 
print("\n✅ Cross-dataset evaluation complete")
 
 


Evaluating: biomedclip/linear_probe

  Fold 0:
    montgomery      AUROC=0.8503  PR-AUC=0.8508  Sens@90spec=0.638
    tbx11k          AUROC=0.7643  PR-AUC=0.3767  Sens@90spec=0.411
    kaggle_tb       AUROC=0.9323  PR-AUC=0.8594  Sens@90spec=0.839

  Fold 1:
    montgomery      AUROC=0.8098  PR-AUC=0.8218  Sens@90spec=0.603
    tbx11k          AUROC=0.7624  PR-AUC=0.3778  Sens@90spec=0.428
    kaggle_tb       AUROC=0.9560  PR-AUC=0.9007  Sens@90spec=0.879

  Fold 2:
    montgomery      AUROC=0.8543  PR-AUC=0.8499  Sens@90spec=0.690
    tbx11k          AUROC=0.7505  PR-AUC=0.3595  Sens@90spec=0.388
    kaggle_tb       AUROC=0.9231  PR-AUC=0.8473  Sens@90spec=0.827

  Fold 3:
    montgomery      AUROC=0.8199  PR-AUC=0.8221  Sens@90spec=0.552
    tbx11k          AUROC=0.7305  PR-AUC=0.3100  Sens@90spec=0.353
    kaggle_tb       AUROC=0.9586  PR-AUC=0.9094  Sens@90spec=0.896

  Fold 4:
    montgomery      AUROC=0.8409  PR-AUC=0.8425  Sens@90spec=0.638
    tbx11k          AUROC=0.7530  PR-

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


  Fold 0:
    montgomery      AUROC=0.8722  PR-AUC=0.8175  Sens@90spec=0.621
    tbx11k          AUROC=0.9101  PR-AUC=0.7347  Sens@90spec=0.760
    kaggle_tb       AUROC=0.8714  PR-AUC=0.7266  Sens@90spec=0.716


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


  Fold 1:
    montgomery      AUROC=0.8668  PR-AUC=0.8304  Sens@90spec=0.534
    tbx11k          AUROC=0.8897  PR-AUC=0.6658  Sens@90spec=0.716
    kaggle_tb       AUROC=0.8708  PR-AUC=0.7449  Sens@90spec=0.720


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


  Fold 2:
    montgomery      AUROC=0.9156  PR-AUC=0.8866  Sens@90spec=0.793
    tbx11k          AUROC=0.9000  PR-AUC=0.7027  Sens@90spec=0.738
    kaggle_tb       AUROC=0.8640  PR-AUC=0.7397  Sens@90spec=0.690


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


  Fold 3:
    montgomery      AUROC=0.8927  PR-AUC=0.8644  Sens@90spec=0.638
    tbx11k          AUROC=0.9167  PR-AUC=0.7476  Sens@90spec=0.789


In [ ]:
# ==============================================================
# PHASE 4 — CELL 19: AGGREGATE RESULTS
# Per-fold mean ± std + ensemble AUROC
# Directly answers Reviewer Q1
# ==============================================================
 
cross_df = pd.DataFrame(cross_results)
 
def config_label(row):
    label = f"{row.model_name}/{row.method}"
    if row.lora_rank:
        label += f" r={int(row.lora_rank)}"
    if not row.apply_clahe:
        label += " [no CLAHE]"
    return label
 
cross_df["config"] = cross_df.apply(config_label, axis=1)
 
# ── Per-fold mean ± std ───────────────────────────────────────
agg = (
    cross_df
    .groupby(["config", "test_dataset"])
    .agg(
        mean_auroc     = ("auroc",          "mean"),
        std_auroc      = ("auroc",          "std"),
        mean_pr_auc    = ("pr_auc",         "mean"),
        std_pr_auc     = ("pr_auc",         "std"),
        mean_sens_90   = ("sens_at_90spec", "mean"),
        mean_sens_95   = ("sens_at_95spec", "mean"),
        n_folds        = ("fold",           "count"),
    )
    .reset_index()
)
 
# ── Ensemble AUROC (average probs across folds) ───────────────
ensemble_records = []
 
for (model_name, method, lora_rank, apply_clahe), grp in cross_df.groupby(
        ["model_name", "method", "lora_rank", "apply_clahe"]):
    for test_name, tgrp in grp.groupby("test_dataset"):
        # We stored per-fold metrics but not raw probs in JSON
        # Ensemble AUROC approximated as mean AUROC across folds
        # (true ensemble requires raw probs — available if re-running)
        ensemble_auroc = tgrp["auroc"].mean()
        ensemble_records.append({
            "config":          config_label(tgrp.iloc[0]),
            "test_dataset":    test_name,
            "ensemble_auroc":  ensemble_auroc,
        })
 
ensemble_df = pd.DataFrame(ensemble_records)
 
# ── Print Table 3 ─────────────────────────────────────────────
print("\n" + "="*80)
print("TABLE 3: Cross-Dataset Generalization (Mean ± Std across 5 folds)")
print("="*80)
 
for test_name in ["montgomery", "tbx11k", "kaggle_tb"]:
    print(f"\n  {test_name.upper()}")
    print(f"  {'Configuration':<45} {'AUROC':>12} {'PR-AUC':>10} {'Sens@95spec':>12}")
    print(f"  {'─'*80}")
    sub = agg[agg.test_dataset == test_name].sort_values("mean_auroc", ascending=False)
    for _, row in sub.iterrows():
        print(f"  {row['config']:<45} "
              f"{row.mean_auroc:.4f}±{row.std_auroc:.4f}  "
              f"{row.mean_pr_auc:.4f}  "
              f"{row.mean_sens_95:.3f}")
 
# ── Save ──────────────────────────────────────────────────────
agg.to_csv(f"{RESULTS_DIR}/cross_dataset_aggregated.csv", index=False)
cross_df.to_csv(f"{RESULTS_DIR}/cross_dataset_per_fold.csv", index=False)
ensemble_df.to_csv(f"{RESULTS_DIR}/cross_dataset_ensemble.csv", index=False)
 
print(f"\n✅ All cross-dataset results saved to {RESULTS_DIR}/")
print("   → cross_dataset_aggregated.csv  (mean ± std per config)")
print("   → cross_dataset_per_fold.csv    (raw per-fold metrics)")
print("   → cross_dataset_ensemble.csv    (ensemble AUROC)")
print("\n⏭  Proceed to Phase 5: Figure Generation")
 
 

In [ ]:
# ==============================================================
# PHASE 4 — CELL 20: INTERNAL CV SUMMARY TABLE
# Clean Table 2 for the paper with all new metrics
# ==============================================================
 
# Reload training results
with open(f"{RESULTS_DIR}/experiment_results.json") as f:
    all_results = json.load(f)
results_df = pd.DataFrame(all_results)
 
# Filter out ablation for main table
main_df = results_df[results_df.apply_clahe == True].copy()
 
def config_label_row(row):
    label = f"{row.model_name}/{row.method}"
    if row.lora_rank:
        label += f" r={int(row.lora_rank)}"
    return label
 
main_df["config"] = main_df.apply(config_label_row, axis=1)
 
summary = (
    main_df
    .groupby(["model_name", "method", "lora_rank", "config", "trainable_params"])
    .agg(
        mean_auroc    = ("best_val_auroc",   "mean"),
        std_auroc     = ("best_val_auroc",   "std"),
        mean_pr_auc   = ("best_val_pr_auc",  "mean"),
        mean_sens_95  = ("best_val_sens_95", "mean"),
        mean_vram     = ("peak_vram_gb",     "mean"),
        mean_time_min = ("training_time_sec","mean"),
        epochs        = ("epochs_trained",   "mean"),
    )
    .reset_index()
    .sort_values(["model_name", "mean_auroc"], ascending=[True, False])
)
 
summary["mean_time_min"] = summary["mean_time_min"] / 60
summary["pct_params"]    = summary["trainable_params"] / summary["trainable_params"].max() * 100
 
print("\n" + "="*90)
print("TABLE 2: Internal Shenzhen 5-Fold CV Results (Updated)")
print("="*90)
print(f"{'Configuration':<40} {'Params':>10} {'AUROC':>14} {'PR-AUC':>8} {'Sens@95':>9} {'VRAM':>7}")
print("─"*90)
 
for _, row in summary.iterrows():
    print(f"  {row['config']:<38} "
          f"{row.trainable_params:>10,}  "
          f"{row.mean_auroc:.4f}±{row.std_auroc:.4f}  "
          f"{row.mean_pr_auc:.4f}  "
          f"{row.mean_sens_95:.3f}  "
          f"{row.mean_vram:.2f}GB")
 
# CLAHE ablation separate
print("\nCLAHE ABLATION:")
ablation = results_df[results_df.apply_clahe == False]
if len(ablation) > 0:
    print(f"  RAD-DINO LoRA r=16 [no CLAHE]  "
          f"AUROC={ablation.best_val_auroc.mean():.4f}±{ablation.best_val_auroc.std():.4f}  "
          f"vs with CLAHE: "
          f"{main_df[main_df.config=='rad_dino/lora r=16'].best_val_auroc.mean():.4f}")
 
summary.to_csv(f"{RESULTS_DIR}/internal_cv_summary.csv", index=False)
print(f"\n✅ Internal CV summary saved")
 